In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import zscore
import os

# 设置文件路径为"charge2"同级目录
input_dir = os.path.join(os.getcwd(), 'charge2')
output_dir = input_dir  # 结果保存到 "charge2" 文件夹

# 读取 SOH_Result.xlsx 文件
file_path = os.path.join(input_dir, 'SOH_Result.xlsx')
df = pd.read_excel(file_path, engine='openpyxl')

# 提取第二列（y）和第四列（x）
y = df.iloc[:, 1].values  # 第二列
x = df.iloc[:, 3].values  # 第四列

# 动态计算第四列（累计里程）的最小值
min_mileage = x.min()

# 计算 z-scores
z_scores = np.abs(zscore(np.vstack([x, y]).T))

# 设置阈值，删除离群点（通常 z-score > 3 被认为是离群点）
threshold = 2
filtered_entries = (z_scores < threshold).all(axis=1)

x_filtered = x[filtered_entries]
y_filtered = y[filtered_entries]

# 应用 RLowess 方法进行平滑
smoothed = lowess(y_filtered, x_filtered, frac=0.6, it=15)  # frac 是平滑参数，it 是迭代次数
# 提取平滑后的 x 和 y 值
x_smooth = smoothed[:, 0]
y_smooth = smoothed[:, 1]

# 绘制原始数据、去除离群点后的数据和平滑数据
plt.figure(figsize=(10, 4))
plt.scatter(x, y, label='Original Data', color='gray', alpha=0.3)
plt.scatter(x_filtered, y_filtered, label='Filtered Data', color='blue', alpha=0.7)
plt.plot(x_smooth, y_smooth, label='RLowess Smoothed Data', color='red')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('RLowess Smoothing with Outlier Removal')
# 指定Y轴范围
plt.ylim([130, 160])
plt.legend()
plt.show()

# 调整数据，动态使用最小里程值进行替换
x_adjusted = x - min_mileage
x_filtered_adjusted = x_filtered - min_mileage
x_smooth_adjusted = x_smooth - min_mileage
y_adjusted = y / 1.5
y_filtered_adjusted = y_filtered / 1.5
y_smooth_adjusted = y_smooth / 1.5

# 绘图 - 第二张图：调整后的数据
plt.figure(figsize=(8, 4))
plt.rcParams['font.family'] = 'Times New Roman'  # 设置字体为新罗马

# 绘制不同数据点和曲线
plt.scatter(x_adjusted, y_adjusted, label='Outliers', color='gray', alpha=0.3)
plt.scatter(x_filtered_adjusted, y_filtered_adjusted, label='Filtered Data', color='blue', alpha=0.7)
plt.plot(x_smooth_adjusted, y_smooth_adjusted, label='RLOESS Smoothed Data', color='red', linewidth=3)

# 图表标签和标题
plt.xlabel('Mileage increment (km)', fontsize=16)
plt.ylabel('SOH(%)', fontsize=14)

# 设置X轴和Y轴的数字刻度大小和粗细
plt.tick_params(axis='x', labelsize=14, width=2)
plt.tick_params(axis='y', labelsize=14, width=2)

# 指定Y轴范围
plt.ylim([85, 100])
plt.legend()

# 设置图例字体大小
plt.legend(prop={'size': 14})
plt.show()

# 将平滑后的 x 和 y 值存储到新的 DataFrame 中
smoothed_df = pd.DataFrame({'x': x_smooth, 'y': y_smooth})

# 保存到新的 Excel 文件
smoothed_file_path = os.path.join(output_dir, 'SOH_rlowess.xlsx')
smoothed_df.to_excel(smoothed_file_path, index=False, engine='openpyxl')

print(f"Processing completed. Results saved to {smoothed_file_path}.")
